# Частота цифры и перебор вариантов

Почтовый сервис сканирует индексы: каждая картинка 8×8 — одна цифра. Прежде чем распознавать, надо знать, что вообще приходит на вход.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

import matplotlib.pyplot as plt


def show_digit(row, title=''):
    """Одна строка таблицы -> картинка 8x8."""
    values = [int(v) for v in row[PIXELS]]
    grid = [values[i * 8:(i + 1) * 8] for i in range(8)]
    plt.imshow(grid, cmap='gray_r')
    plt.title(title)
    plt.axis('off')


## 1. Размер таблицы

Сколько картинок, сколько пикселей в одной картинке, сколько разных цифр.

**How:** `df.shape`, `len(PIXELS)`, `df['label'].nunique()`.

In [ ]:
n_images = None
n_pixels = None
n_classes = None
assert n_images is not None and n_pixels is not None and n_classes is not None
assert n_images == 1797
assert n_pixels == 64 and n_classes == 10
print(n_images, n_pixels, n_classes)

## 2. Одна картинка — это 64 числа

Нарисуйте строку 17: `show_digit(df.loc[17])`. Посчитайте, сколько пикселей в этой строке равны 0 (белый фон) -> `n_zero_17`.

**Вопрос:** почему пустых пикселей так много?

In [ ]:
# show_digit(df.loc[17], title='строка 17')
n_zero_17 = None
assert n_zero_17 is not None
assert 0 < int(n_zero_17) < 64
print('нулевых пикселей:', n_zero_17)

## 3. Частота цифры = её вероятность в потоке

`class_counts` — сколько картинок каждой цифры; `class_share` — доля каждой цифры (частота, делённая на общее число).

**How:** `value_counts()` и `value_counts(normalize=True)`.

**Checkpoint:** почему сумма долей равна 1?

In [ ]:
class_counts = None
class_share = None
assert class_counts is not None and class_share is not None
assert len(class_share) == 10
assert abs(float(class_share.sum()) - 1.0) < 1e-9
print(class_share.sort_index().round(3))

## 4. Baseline: всегда отвечать самой частой цифрой

Самая частая цифра -> `top_digit`; её доля -> `baseline_accuracy` (так часто угадает распознаватель, который всегда отвечает одно и то же).

**Зачем:** любую модель сравниваем с этим числом, иначе «90% точности» ни о чём не говорит.

In [ ]:
top_digit = None
baseline_accuracy = None
assert top_digit is not None and baseline_accuracy is not None
assert int(top_digit) in range(10)
assert 0.09 < float(baseline_accuracy) < 0.12
print(top_digit, round(float(baseline_accuracy), 3))

## 5. Вероятность события «1 или 7»

Индексы часто путают 1 и 7. Какова доля картинок, где цифра — 1 **или** 7? -> `p_1_or_7`.

**How:** сложить две доли или посчитать долю строк по условию `isin([1, 7])`.

In [ ]:
p_1_or_7 = None
assert p_1_or_7 is not None
assert 0.15 < float(p_1_or_7) < 0.25
print(round(float(p_1_or_7), 3))

## 6. Сколько пар пикселей можно сравнить

Сколько существует **пар разных** пикселей из 64 (порядок не важен)? Посчитайте **перебором** двумя вложенными циклами -> `n_pairs`.

Сверьте с формулой n·(n−1)/2 -> `n_pairs_formula`.

**Связь:** так же перебираются варианты настроек распознавателя.

In [ ]:
n_pairs = 0
# for i in range(len(PIXELS)):
#     for j in range(...):
#         n_pairs += 1
n_pairs_formula = None
assert n_pairs == 2016
assert n_pairs_formula is not None and int(n_pairs_formula) == n_pairs
print(n_pairs)

## 7. Сколько настроек надо проверить (правило произведения)

Соберите список `configs` из всех сочетаний: число соседей из `K_VALUES` и набор признаков из `FEATURE_SETS` (перебор двумя циклами, каждый элемент — кортеж).

**Правило произведения:** вариантов столько, сколько произведение длин.

In [ ]:
K_VALUES = [1, 3, 5, 7, 9]
FEATURE_SETS = ['все 64 пикселя', 'верхняя половина', 'сумма яркости']
configs = []
# соберите кортежи (k, набор)
assert len(configs) == len(K_VALUES) * len(FEATURE_SETS)
assert all(isinstance(c, tuple) and len(c) == 2 for c in configs)
print(len(configs), configs[:3])

## 8. Эксперимент: поток изменился

Представьте участок, где почти все индексы начинаются с 0 и 1. Возьмите только строки с цифрами 0 и 1 -> `stream`; посчитайте новый `baseline_stream` (доля самой частой цифры там).

Ответьте в `BIAS_NOTE`: почему распознаватель, оценённый на всей таблице, может вести себя иначе на таком участке. **Готового ответа нет.**

In [ ]:
stream = None
baseline_stream = None
BIAS_NOTE = ''
assert stream is not None and baseline_stream is not None
assert float(baseline_stream) > 0.4
assert len(BIAS_NOTE) > 40
print(len(stream), round(float(baseline_stream), 3), BIAS_NOTE)

## 9. Расширение: две картинки подряд

Если две картинки берут независимо, вероятность, что они **одной** цифры, — сумма квадратов долей. Посчитайте `p_same` и сравните с 1/10.

**Вопрос:** почему это не то же самое, что «вероятность угадать»?

In [ ]:
p_same = None
assert p_same is not None
assert 0.09 < float(p_same) < 0.12
print(round(float(p_same), 4))